In [144]:
import json
import networkx as nx
from itertools import combinations
from collections import defaultdict
from pathlib import Path
from pyvis.network import Network
import os, sys

import numpy as np
import pyLDAvis
import pyLDAvis.lda_model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation



from networkx.algorithms.community import louvain_communities
from networkx.algorithms.community import greedy_modularity_communities


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.makedirs(PROJECT_ROOT / "graph_artifacts" / "visuals", exist_ok=True)
sys.path.append(str(PROJECT_ROOT))

from utils.utils import load, dump

PROCESSED_VIDEO_DATA_PATH = PROJECT_ROOT / "data" / "video_data_processed.json"
OUTVISUAL = PROJECT_ROOT / "data" / "GraphVisual.html" 
OUTPUTGRAPH = PROJECT_ROOT / "data" / "graph.graphml"

ARTIFACT_DIR = PROJECT_ROOT / "graph_artifacts"
VISUAL_DIR  = ARTIFACT_DIR / "visuals"

In [145]:
# Graph Helper Class
class graph:
    
# INIT ===========================================================================================================

    def __init__(self, Gr = None, gPath = None):
        self.G = nx.MultiDiGraph()
        self.G = Gr
        if gPath:
            try:
                self.G = nx.read_graphml(path=gPath)
            except FileNotFoundError as e:
                print(f"Tried reading graph from memory, failed. Graph doesn't exist yet.")

# VISUALIZE ======================================================================================================================

    def visualize(self, outPath=OUTVISUAL):
        # Get top nodes
        top_nodes = sorted(self.G.degree(), key=lambda x: x[1], reverse=True)[:1000]
        G_sub = self.G.subgraph([n for n, d in top_nodes])

        net = Network(height="750px", width="100%", directed=False, notebook=False)
        net.from_nx(G_sub)

        # Colour by node type
        for node in net.nodes:
            if self.G.nodes[node["id"]].get("node_type") == "celebrity":
                node["color"] = "#e74c3c"   # celebs
            elif self.G.nodes[node["id"]].get("node_type") == "brand":
                node["color"] = "#3498db"   # brands
            else:
                node["color"] = "#95a5a6"   # else
                
        for edge in net.edges:
            edge["label"] = ""
            edge["title"] = ""
            edge["width"] = 0.5
            edge["color"] = "#cccccc"

        net.set_options("""
        {
        "edges": {
            "arrows": { "to": { "enabled": false } },
            "color": { "color": "#cccccc", "opacity": 0.75 },
            "width": 0.5,
            "smooth": { "enabled": false }
        },
        "physics": {
            "forceAtlas2Based": {
            "gravitationalConstant": -50,
            "springLength": 100
            },
            "solver": "forceAtlas2Based",
            "stabilization": {
                "enabled": true,
                "iterations": 200,
                "fit": true
                }
            }
        }
        """)

        net.show(str(outPath), notebook=False)
        print(f"Graph saved to: {outPath}")

# RUN COMMUNITY DETECTION ======================================================================================================== 

    def communities(self, method="louvain", resolution=1.0, min_weight = 20):
        G_undirected = self.G.to_undirected() if self.G.is_directed() else self.G


        # Filter Edge weights
        if min_weight > 1:
                edges_to_remove = [(u, v) for u, v, d in G_undirected.edges(data=True)
                                if d.get("width", 0) < min_weight]
                G_undirected = G_undirected.copy()
                G_undirected.remove_edges_from(edges_to_remove)
                G_undirected.remove_nodes_from(list(nx.isolates(G_undirected)))
                print(f"After weight filter, Nodes: {G_undirected.number_of_nodes()}, Edges: {G_undirected.number_of_edges()}")

        if G_undirected.number_of_nodes() == 0:
            print(f"Graph is empty after min_weight={min_weight} filter — try a lower threshold.")
            return []


        if method == "louvain":
            result = louvain_communities(G_undirected, seed=42, resolution=resolution)

        elif method == "greedy":
            result = greedy_modularity_communities(G_undirected)

        else:
            #Default to louvian
            result = louvain_communities(G_undirected, seed=42, resolution=resolution)


        # Q modularity for Communities accuraccy score 
        modularity = nx.community.modularity(G_undirected, result)

        # Tag every node with its community ID
        for i, community in enumerate(result):
            for node in community:
                self.G.nodes[node]["community"] = i

        self.communities_result = result

        print(f"Communities found:  {len(result)}")
        print(f"Modularity (Q):     {modularity:.4f}")
        print(f"Largest community:  {max(len(c) for c in result)} nodes")
        print(f"Smallest community: {min(len(c) for c in result)} nodes")
        print()
        
        for i, community in enumerate(result):
            celebs = [n for n in community if self.G.nodes[n].get("node_type") == "celebrity"]
            brands = [n for n in community if self.G.nodes[n].get("node_type") == "brand"]
            print(f"  Community {i}: {len(community)} nodes | {len(celebs)} celebs | {len(brands)} brands")
            print(f"    Top members: {sorted(community, key=lambda n: self.G.nodes[n].get('mention_count', 0), reverse=True)[:5]}")

        return result

# Degree centrality ==============================================================================================================

    def degree_centrality(self, outPath=ARTIFACT_DIR / "degree_centrality.json"):
        scores = nx.degree_centrality(self.G)
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores

# betweenness centrality ==========================================================================================================

    def betweenness_centrality(self, outPath=ARTIFACT_DIR / "betweenness_centrality.json"):
        scores = nx.betweenness_centrality(self.G,  weight="co_mention_count")
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores
    
# closeness centrality ===========================================================================================================

    def closeness_centrality(self, outPath=ARTIFACT_DIR / "closeness_centrality.json"):
        scores = nx.closeness_centrality(self.G)
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores
    
# Eigenvector centrality ===========================================================================================================

    def eigenvector_centrality(self, outPath=ARTIFACT_DIR / "eigenvector_centrality.json"):
        try:
            scores = nx.eigenvector_centrality(self.G, weight="co_mention_count", max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            print("Eigenvector centrality failed to converge, trying with increased iterations...")
            scores = nx.eigenvector_centrality_numpy(self.G, weight="co_mention_count")
            
        sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

        for i, (node, score) in enumerate(list(sorted_scores.items())[:20], start=1):
            mention_count = self.G.nodes[node].get("mention_count", 0)
            node_type = self.G.nodes[node].get("node_type", "unknown")
            print(f"[{i:02}] {node:<30} {score:.4f}  |  mentions: {mention_count}  |  type: {node_type}")

        dump(sorted_scores, outPath=outPath)
        return sorted_scores

# EXPORT =========================================================================================================================

    def export(self, outPath = OUTPUTGRAPH):
        nx.write_graphml(self.G, outPath)

In [146]:
class ldaTopics:
    # Parameters ------------------------------------------------------------------------------------------

    def __init__(self):

        self.NUM_TOPICS = 5
        self.WORDS_PER_TOPIC = 20
        self.FEATURE_NUM = 1000
        self.ITERATIONS = 100

    # LDA --------------------------------------------------------------------------------------------------

    def buildCorpus(self, comments: list):
        """
        Returns list of comment corpus
        @params list: List of comments object
        returns list of corpus
        """
        self.corpus = []
        for comment in comments:
            self.corpus.append(" ".join(comment))
        return self.corpus

    def display_topics(self, model, featureNames, numTopWords):
        """
        Prints out the most associated words for each topic.

        @param model: lda model.
        @param featureNames: list of strings, representing the list of features/words.
        @param numTopWords: number of words to print per topic.
        """

        # print out the topic distributions
        for topicId, lTopicDist in enumerate(model.components_):
            print('Topic %d:' % (topicId + 1))
            print(' '.join([featureNames[i] for i in lTopicDist.argsort()[:-numTopWords - 1:-1]]))

    def ldaAnalysis(self, comments: list, outPath: Path):

        """
        Does LDA topic modeling 

        @params tokens: list of tokens

        outputs visualisation to .html file
        
        """


        print(f"Building corpus...")

        corpus = self.buildCorpus(comments)

        print(f"Built corpus from {len(comments)} comments.")

        # Counter Vectorizer

        print(f"fitting counter vectorizer")

        tfVectorizer = CountVectorizer(max_df=0.90, min_df=5, max_features=self.FEATURE_NUM, stop_words='english')
        tf = tfVectorizer.fit_transform(corpus)
        # extract the names of the features (in our case, the words)
        tfFeatureNames = tfVectorizer.get_feature_names_out()

        print(f"Done fitting CountVectorizer")

        #
        # LDA MODEL
        #

        print(f"Fitting LDA Model.")

        ldaModel = LatentDirichletAllocation(n_components=self.NUM_TOPICS, max_iter=self.ITERATIONS, learning_method='online').fit(tf)

        print(f"Done!")


        # Display Topics 

        print("Displaying TOPICS...")
        self.display_topics(ldaModel, tfFeatureNames, self.WORDS_PER_TOPIC)

        # Visualize and save Topic visual

        print("Visualizing Topics")
        panel = pyLDAvis.lda_model.prepare(
            ldaModel,
            tf,
            tfVectorizer,
            mds="mmds",
            n_jobs=1 
        )

        pyLDAvis.save_html(panel, str(outPath))

        print(f"Saved visual successfully!")

        return True







In [147]:
def co_mention_comments(comments, celeb_list, target=1000, max_stepdowns=None):
    collected_ids  = set()
    matched_combos = []

    num_celebs = len(celeb_list)
    floor = num_celebs - max_stepdowns if max_stepdowns is not None else 2


    while num_celebs >= floor and len(collected_ids) < target:
        combos     = list(combinations(celeb_list, num_celebs))
        newly_found = 0

        if num_celebs == 1:
            break
        
        for combo in combos:
            combo_set  = set(combo)

            for comment in comments:
                comment_celebs = set(comment.get("celebs", []))
                comment_id     = comment.get("comment_id")

                if comment_id in collected_ids:
                    continue
                if combo_set.issubset(comment_celebs):
                    collected_ids.add(comment_id)
                    newly_found += 1

                if len(collected_ids) >= target:
                    break

            if len(collected_ids) >= target:
                break


        print(f"  num_celebs={num_celebs} ({len(combos)} combos) found {newly_found} new comments | total: {len(collected_ids)}")
        matched_combos.append({"n_celebs": num_celebs, "newly_found": newly_found})
        num_celebs -= 1

    print(f"\nDone. Total comment IDs collected: {len(collected_ids)}")
    return list(collected_ids), matched_combos


In [148]:
with open(PROCESSED_VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    processed = json.load(f)

comments = processed["comments"]
entity_counts = processed["entity_counts"]

In [149]:
co_mentions = defaultdict(int)
entity_sentiment = defaultdict(list)

# GET CELEBRITY CO-MENTIONS 
for comment in comments:
    entities = list(set(comment.get("celebs", [])))

    if len(entities) < 2:
        continue

    for pair in combinations(sorted(entities), 2):
        co_mentions[pair] += 1

G1 = nx.Graph()
print(f"Building Graph 1 Celebrity co-mentions:")
print("-"*82)
# Add nodes
for celeb, count in entity_counts["celebs"].items():
    G1.add_node(celeb, node_type="celebrity", mention_count=count)

# Add edges
MIN_CO_MENTIONS = 20
for (e1, e2), weight in co_mentions.items():
    if weight >= MIN_CO_MENTIONS:
        G1.add_edge(e1, e2, weight=weight)

# No co-mentiones with anyone
G1.remove_nodes_from(list(nx.isolates(G1)))

print(f"Nodes:          {G1.number_of_nodes()}")
print(f"Edges:          {G1.number_of_edges()}")
print(f"Celebrities:    {sum(1 for _, d in G1.nodes(data=True) if d.get('node_type') == 'celebrity')}")
print("-"*82)


Building Graph 1 Celebrity co-mentions:
----------------------------------------------------------------------------------
Nodes:          30
Edges:          43
Celebrities:    30
----------------------------------------------------------------------------------


In [150]:
co_mentions = defaultdict(int)

print("Building Graph 2 Celebrity - Brand co-mention")
print("-"*82)

for comment in comments:
    celebs = list(set(comment.get("celebs", [])))
    brands = list(set(comment.get("brands", [])))

    for celeb in celebs:
        for brand in brands:
            co_mentions[(celeb, brand)] += 1

G2 = nx.Graph()

for celeb, count in entity_counts["celebs"].items():
    G2.add_node(celeb, node_type="celebrity", mention_count=count)

for brand, count in entity_counts["brands"].items():
    G2.add_node(brand, node_type="brand", mention_count=count)

MIN_CO_MENTIONS = 2
for (celeb, brand), weight in co_mentions.items():
    if weight >= MIN_CO_MENTIONS:
        G2.add_edge(celeb, brand, weight=weight)

G2.remove_nodes_from(list(nx.isolates(G2)))

print(f"Nodes:       {G2.number_of_nodes()}")
print(f"Edges:       {G2.number_of_edges()}")
print(f"Celebrities: {sum(1 for _, d in G2.nodes(data=True) if d.get('node_type') == 'celebrity')}")
print(f"Brands:      {sum(1 for _, d in G2.nodes(data=True) if d.get('node_type') == 'brand')}")
print("-"*82)

Building Graph 2 Celebrity - Brand co-mention
----------------------------------------------------------------------------------
Nodes:       83
Edges:       120
Celebrities: 51
Brands:      32
----------------------------------------------------------------------------------


In [151]:
CG = graph(G1)

# EXPORT CELEB GRAPH
celebVisual = VISUAL_DIR / "celebs.html"
celebGraph = ARTIFACT_DIR / "celebs.graphml"
CG.visualize(outPath=celebVisual)
CG.export(outPath=celebGraph)

c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\celebs.html
Graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\celebs.html


In [152]:
print(f"Calculating degree centrality on Celebrity Graph")
print("-"*89)
CG.degree_centrality()
print("-"*89)

Calculating degree centrality on Celebrity Graph
-----------------------------------------------------------------------------------------
[01] Beyonce                        0.5862  |  mentions: 1329  |  type: celebrity
[02] LISA                           0.2759  |  mentions: 1045  |  type: celebrity
[03] Jisoo                          0.2069  |  mentions: 1299  |  type: celebrity
[04] Rose                           0.2069  |  mentions: 938  |  type: celebrity
[05] Emma Chamberlain               0.1724  |  mentions: 367  |  type: celebrity
[06] JENNIE                         0.1724  |  mentions: 407  |  type: celebrity
[07] Ningning                       0.1724  |  mentions: 179  |  type: celebrity
[08] Karina                         0.1724  |  mentions: 122  |  type: celebrity
[09] Madonna                        0.1034  |  mentions: 434  |  type: celebrity
[10] Sabrina Carpenter              0.1034  |  mentions: 200  |  type: celebrity
[11] Cher                           0.0690  |  m

In [153]:
print(f"Calculating betweenness centrality on Celebrity Graph")
print("-"*89)
CG.betweenness_centrality()
print("-"*89)

Calculating betweenness centrality on Celebrity Graph
-----------------------------------------------------------------------------------------
[01] Beyonce                        0.5538  |  mentions: 1329  |  type: celebrity
[02] LISA                           0.1351  |  mentions: 1045  |  type: celebrity
[03] Jisoo                          0.0903  |  mentions: 1299  |  type: celebrity
[04] Emma Chamberlain               0.0632  |  mentions: 367  |  type: celebrity
[05] Rose                           0.0567  |  mentions: 938  |  type: celebrity
[06] Madonna                        0.0567  |  mentions: 434  |  type: celebrity
[07] Karan Johar                    0.0025  |  mentions: 203  |  type: celebrity
[08] Cher                           0.0000  |  mentions: 107  |  type: celebrity
[09] Ciara                          0.0000  |  mentions: 110  |  type: celebrity
[10] Rihanna                        0.0000  |  mentions: 742  |  type: celebrity
[11] Nicole Kidman                  0.0000 

In [154]:
print(f"Calculating closeness centrality on Celebrity Graph")
print("-"*89)
CG.closeness_centrality()
print("-"*89)

Calculating closeness centrality on Celebrity Graph
-----------------------------------------------------------------------------------------
[01] Beyonce                        0.6207  |  mentions: 1329  |  type: celebrity
[02] LISA                           0.4844  |  mentions: 1045  |  type: celebrity
[03] Jisoo                          0.4514  |  mentions: 1299  |  type: celebrity
[04] Emma Chamberlain               0.4414  |  mentions: 367  |  type: celebrity
[05] Sabrina Carpenter              0.4226  |  mentions: 200  |  type: celebrity
[06] Madonna                        0.3820  |  mentions: 434  |  type: celebrity
[07] Cher                           0.3748  |  mentions: 107  |  type: celebrity
[08] Heidi Klum                     0.3748  |  mentions: 337  |  type: celebrity
[09] Ciara                          0.3611  |  mentions: 110  |  type: celebrity
[10] Rihanna                        0.3611  |  mentions: 742  |  type: celebrity
[11] Nicole Kidman                  0.3611  |

In [155]:
print(f"Calculating eigenvector centrality on Celebrity Graph")
print("-"*89)
CG.eigenvector_centrality()
print("-"*89)

Calculating eigenvector centrality on Celebrity Graph
-----------------------------------------------------------------------------------------
[01] LISA                           0.4254  |  mentions: 1045  |  type: celebrity
[02] Beyonce                        0.3813  |  mentions: 1329  |  type: celebrity
[03] Jisoo                          0.3674  |  mentions: 1299  |  type: celebrity
[04] Rose                           0.3182  |  mentions: 938  |  type: celebrity
[05] JENNIE                         0.3096  |  mentions: 407  |  type: celebrity
[06] Ningning                       0.3096  |  mentions: 179  |  type: celebrity
[07] Karina                         0.3096  |  mentions: 122  |  type: celebrity
[08] Emma Chamberlain               0.2018  |  mentions: 367  |  type: celebrity
[09] Sabrina Carpenter              0.1804  |  mentions: 200  |  type: celebrity
[10] Heidi Klum                     0.1043  |  mentions: 337  |  type: celebrity
[11] Madonna                        0.0859 

In [156]:
CBG = graph(G2)

# EXPORT CELEB - BRAND GRAPH 
celebBrandVisual = VISUAL_DIR / "CBG.html"
celebBrandGraph = ARTIFACT_DIR / "CBG.graphml"
CBG.visualize(outPath=celebBrandVisual)
CBG.export(outPath=celebBrandGraph)



c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\CBG.html
Graph saved to: c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\graph_artifacts\visuals\CBG.html


In [157]:
celebCommunities = CG.communities(method="greedy", min_weight=32)
communities_serializable = [list(community) for community in celebCommunities]
dump(communities_serializable, outPath=ARTIFACT_DIR / "celebCommunities.json")

print(f"celebCommnities: {celebCommunities[:10]}")

After weight filter, Nodes: 15, Edges: 18
Communities found:  3
Modularity (Q):     0.4707
Largest community:  7 nodes
Smallest community: 2 nodes

  Community 0: 7 nodes | 7 celebs | 0 brands
    Top members: ['Beyonce', 'Rihanna', 'Madonna', 'Emma Chamberlain', 'Sabrina Carpenter']
  Community 1: 6 nodes | 6 celebs | 0 brands
    Top members: ['Jisoo', 'LISA', 'Rose', 'JENNIE', 'Ningning']
  Community 2: 2 nodes | 2 celebs | 0 brands
    Top members: ['Karan Johar', 'Isha Ambani']
celebCommnities: [frozenset({'Emma Chamberlain', 'Beyonce', 'Sabrina Carpenter', 'Rihanna', 'Jay-Z', 'Madonna', 'Blue Ivy'}), frozenset({'LISA', 'Karina', 'Jisoo', 'Rose', 'JENNIE', 'Ningning'}), frozenset({'Karan Johar', 'Isha Ambani'})]


In [158]:
# get community comments.

community_comment_list = []

for celebs in celebCommunities:
    comment_ids, matches = co_mention_comments(comments=comments, celeb_list=celebs, target= 1000, max_stepdowns=2)
    community_comment_list.append(comment_ids)



  num_celebs=7 (1 combos) found 0 new comments | total: 0
  num_celebs=6 (7 combos) found 2 new comments | total: 2
  num_celebs=5 (21 combos) found 4 new comments | total: 6

Done. Total comment IDs collected: 6
  num_celebs=6 (1 combos) found 11 new comments | total: 11
  num_celebs=5 (6 combos) found 6 new comments | total: 17
  num_celebs=4 (15 combos) found 116 new comments | total: 133

Done. Total comment IDs collected: 133
  num_celebs=2 (1 combos) found 32 new comments | total: 32

Done. Total comment IDs collected: 32


In [159]:
#map CID to comment tokens
comment_map = {c["comment_id"]: c["comment_tokens_topic"] for c in comments}
lda = ldaTopics()
for i, comment_ids in enumerate(community_comment_list, start=1):

    print(f"COMMUNITY {i}")
    print("-"*82)

    community_tokens = [comment_map[cid] for cid in comment_ids if cid in comment_map]
    
    outPath = VISUAL_DIR / f"lda_community_{i}.html"
    lda.ldaAnalysis(community_tokens, outPath=outPath)


COMMUNITY 1
----------------------------------------------------------------------------------
Building corpus...
Built corpus from 6 comments.
fitting counter vectorizer
Done fitting CountVectorizer
Fitting LDA Model.
Done!
Displaying TOPICS...
Topic 1:
tyla tate doja mcrae ciara cat taylor teyana blue xcx ivy yseult
Topic 2:
blue cat mcrae ivy ciara xcx teyana yseult taylor tyla tate doja
Topic 3:
doja ciara xcx blue tate cat yseult teyana ivy taylor tyla mcrae
Topic 4:
blue xcx tate teyana ciara doja taylor ivy tyla cat mcrae yseult
Topic 5:
tyla taylor ciara mcrae teyana doja cat yseult ivy tate blue xcx
Visualizing Topics


c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


Saved visual successfully!
COMMUNITY 2
----------------------------------------------------------------------------------
Building corpus...
Built corpus from 133 comments.
fitting counter vectorizer
Done fitting CountVectorizer
Fitting LDA Model.
Done!
Displaying TOPICS...
Topic 1:
interview red carpet blinks blackpink think blink normal met new make year literally members wearing blue mean outfits white ate
Topic 2:
ate karina ningning nailed time best stunning outfits hard dresses girls plain ysl look looked event dress mean think met
Topic 3:
like theme art dress really fashion time look beautiful looks met wore looking gala good looked literally comes bird mermaid
Topic 4:
blackpink outfit met gala dress love members year gown better boring look like good slayed art blink people amazing come
Topic 5:
pretty creative girl dress pink dresses normal hair met black mermaid white arms best people outfits year love look gala
Visualizing Topics


c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(


Saved visual successfully!
COMMUNITY 3
----------------------------------------------------------------------------------
Building corpus...
Built corpus from 32 comments.
fitting counter vectorizer
Done fitting CountVectorizer
Fitting LDA Model.
Done!
Displaying TOPICS...
Topic 1:
indian art manish malhotra gala theme culture fashion looks ravi varma raja met indians jaipur carpet review south look birla
Topic 2:
manish malhotra looks review birla look sudha indian met gala ravi reddy art outfits carpet varma jaipur theme ananya indians
Topic 3:
ananya birla indian met malhotra manish carpet indians look gala natasha reddy sudha review outfits looks south culture art jaipur
Topic 4:
art outfits manish malhotra sudha reddy birla ananya indian theme indians fashion south patel mona jaipur natasha culture review looks
Topic 5:
look malhotra manish art patel mona ananya birla jaipur reddy sudha ravi varma raja natasha fashion south indian indians met
Visualizing Topics
Saved visual succes

c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:744: FutureWarning: The default value of `n_init` will change from 4 to 1 in 1.9. To suppress this warning, provide some value of `n_init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:754: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
c:\Users\SL\Desktop\soc-med-NetworkAnalysis\Network-Analysis-Project\.venv\Lib\site-packages\sklearn\manifold\_mds.py:771: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(
